## Starting from the EDA I have done...

In [1]:
### Import the dependecies
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt

In [2]:
## Loading the datasets
fraud_df = pd.read_csv('../data/raw/Fraud_Data.csv')
ip_country_df = pd.read_csv ('../data/raw/IpAddress_to_Country.csv')

In [3]:
# Duplicates
print("Duplicate rows:", fraud_df.duplicated().sum())
print("Duplicate user_ids:", fraud_df['user_id'].duplicated().sum())

Duplicate rows: 0
Duplicate user_ids: 0


In [4]:
# Fix dtypes
fraud_df['signup_time'] = pd.to_datetime(fraud_df['signup_time'])
fraud_df['purchase_time'] = pd.to_datetime(fraud_df['purchase_time'])
fraud_df['ip_address'] = fraud_df['ip_address'].astype('int64')

ip_country_df['lower_bound_ip_address'] = (
    ip_country_df['lower_bound_ip_address'].astype('int64')
)

ip_country_df['upper_bound_ip_address'] = (
    ip_country_df['upper_bound_ip_address'].astype('int64')
)
print(fraud_df.dtypes)
print(ip_country_df.dtypes)

user_id                    int64
signup_time       datetime64[ns]
purchase_time     datetime64[ns]
purchase_value             int64
device_id                 object
source                    object
browser                   object
sex                       object
age                        int64
ip_address                 int64
class                      int64
dtype: object
lower_bound_ip_address     int64
upper_bound_ip_address     int64
country                   object
dtype: object


## 1. Geolocation Merge

In [5]:
# print(fraud_df["ip_address"].dtypes)
print(ip_country_df["lower_bound_ip_address"].dtypes)

int64


In [6]:
# Sort the country table by lower bound (required for merge_asof)
ip_country_df = ip_country_df.sort_values('lower_bound_ip_address').reset_index(drop=True)

# Sort fraud_df by ip_address (also required for merge_asof)
fraud_df_sorted = fraud_df.sort_values('ip_address').reset_index(drop=True)

# Range-based merge: find nearest lower_bound <= ip_address
merged = pd.merge_asof(
    fraud_df_sorted,
    ip_country_df,
    left_on='ip_address',
    right_on='lower_bound_ip_address',
    direction='backward'
)

# Verify the match actually falls within the upper bound too
merged['valid_match'] = merged['ip_address'] <= merged['upper_bound_ip_address']
print(merged['valid_match'].value_counts())

# Where invalid, country should be treated as unknown
merged.loc[~merged['valid_match'], 'country'] = 'Unknown'

fraud_df = merged.drop(columns=['lower_bound_ip_address', 'upper_bound_ip_address', 'valid_match'])
print(fraud_df['country'].value_counts().head(10))

valid_match
True     129146
False     21966
Name: count, dtype: int64
country
United States        58049
Unknown              21966
China                12038
Japan                 7306
United Kingdom        4490
Korea Republic of     4162
Germany               3646
France                3161
Canada                2975
Brazil                2961
Name: count, dtype: int64


In [7]:
## proof if feature called 'country' exist after the merge
print(fraud_df.columns)

Index(['user_id', 'signup_time', 'purchase_time', 'purchase_value',
       'device_id', 'source', 'browser', 'sex', 'age', 'ip_address', 'class',
       'country'],
      dtype='object')


In [8]:
# Fraud rate by country, for countries with meaningful volume
country_fraud = fraud_df.groupby('country')['class'].agg(['mean', 'count'])
country_fraud = (country_fraud[country_fraud['count'] >= 100]) * 100  # filter out tiny, unreliable samples
country_fraud = country_fraud.sort_values('mean', ascending=False)
print(country_fraud.head(15))

                           mean  count
country                               
Ecuador               26.415094  10600
Tunisia               26.271186  11800
Peru                  26.050420  11900
Ireland               22.916667  24000
New Zealand           22.302158  27800
Saudi Arabia          18.939394  26400
Denmark               15.918367  49000
Chile                 15.347722  41700
Greece                14.285714  23100
United Arab Emirates  14.035088  11400
Belgium               13.691932  40900
Egypt                 13.370474  35900
Venezuela             13.147410  25100
Norway                12.972085  60900
Hong Kong             12.951168  47100


In [9]:
print(fraud_df[fraud_df['country'] == 'Unknown']['class'].mean())

0.08572339069471001


#### Findings

Ecuador (26.4%), Tunisia (26.3%), Peru (26.1%)

I am not claiming "people from Ecuador are fraudulent" — that's a lazy and unfair reading. The more defensible interpretation is that transactions where the IP-derived country doesn't match the customer's actual expected location are often fraud indicators — VPN use, proxy IPs, or account takeover from a different region than the legitimate account holder normally uses. The country itself isn't the cause; it's a proxy for "this transaction's origin looks anomalous relative to where this platform's typical customers are."

## 2. Feature Engineering

- Feature engineering is the process of transforming or combining raw columns into new variables that make the underlying pattern easier for a model to detect

In [10]:
## 1. time_since_signup - is the difference between purcahse time
## and signup time - more fraud transactions tend to have a small
## difference between them - hit and run
fraud_df['time_since_signup'] = (fraud_df['purchase_time'] - fraud_df['signup_time']).dt.total_seconds()
fraud_df['time_since_signup'] = fraud_df['time_since_signup'] / 3600
print(fraud_df['time_since_signup'].describe())

fraud_df['time_since_signup_hours'] = fraud_df['time_since_signup'] / 3600


count    151112.000000
mean       1370.008125
std         868.406422
min           0.000278
25%         607.431528
50%        1368.429306
75%        2123.479028
max        2879.992222
Name: time_since_signup, dtype: float64


In [11]:
# 2. hour_of_day AND day_of_week - Exposes cyclical timing patterns a raw timestamp hides.
fraud_df['hour_of_day'] = fraud_df['purchase_time'].dt.hour
fraud_df['day_of_week'] = fraud_df['purchase_time'].dt.dayofweek  # 0=Monday, 6=Sunday
print(fraud_df[['hour_of_day', 'day_of_week']].describe())


         hour_of_day    day_of_week
count  151112.000000  151112.000000
mean       11.521593       3.011819
std         6.912474       2.006203
min         0.000000       0.000000
25%         6.000000       1.000000
50%        12.000000       3.000000
75%        17.000000       5.000000
max        23.000000       6.000000


In [12]:
# 3. Transaction velocity (per user)
# This is the aggregation feature — count of transactions per user_id. 
# Since each row here is one purchase, "velocity" in this dataset is really 
# about how many total purchases this user made (there's no separate rolling time-window 
# log per user beyond what's in the dataset). 
# We'll compute purchase count per user as our velocity proxy:
user_txn_count = fraud_df.groupby('user_id')['user_id'].transform('count')
fraud_df['user_transaction_count'] = user_txn_count
print(fraud_df['user_transaction_count'].value_counts().head(10))

user_transaction_count
1    151112
Name: count, dtype: int64


#### Initial Findings

Investigated transaction velocity as a candidate feature; found each user_id appears exactly once in this dataset, meaning per-user purchase frequency carries no variance and would not contribute predictive signal. Velocity is therefore more meaningful in the creditcard.csv context (or would require additional data such as login attempts/session data not present here).

In [13]:
## 4. Device reusability is the one feature that I want to try to create. let me try it:
device_reuse = fraud_df.groupby('device_id')['device_id'].transform('count')
fraud_df['device_shared_count'] = device_reuse
print(fraud_df['device_shared_count'].value_counts().head(10))

# does shared-device correlate with fraud?
print(fraud_df.groupby('device_shared_count')['class'].mean().head(10))

device_shared_count
1     131781
2      10654
11      1111
12      1080
10       920
13       832
14       798
9        702
15       615
16       576
Name: count, dtype: int64
device_shared_count
1     0.030429
2     0.229022
3     0.244444
4     0.625000
5     0.800000
6     0.827586
7     0.860000
8     0.875000
9     0.888889
10    0.898913
Name: class, dtype: float64


### Key finding

when a device is used by only 1 user (device_shared_count = 1), fraud rate is 3.0% - barely above nothing. But as the same device gets reused across more and more distinct users, fraud rate climbs almost monotonically: 2 users → 22.9%, 4 users → 62.5%, 7 users → 86%, 10 users → 89.9%.

But one thing worth considering is that fraud rate is still real and still meaningful, but the extremely high rates at the tail (86%, 89%) are based on smaller sample sizes than the 3% figure at the bottom.
So I will take caution here.

In [14]:
print(fraud_df['time_since_signup_hours'].describe())
print(fraud_df.groupby('class')['time_since_signup'].describe())

count    1.511120e+05
mean     3.805578e-01
std      2.412240e-01
min      7.716049e-08
25%      1.687310e-01
50%      3.801193e-01
75%      5.898553e-01
max      7.999978e-01
Name: time_since_signup_hours, dtype: float64
          count         mean         std       min         25%          50%  \
class                                                                         
0      136961.0  1441.994052  830.163558  0.038056  719.119167  1443.030833   
1       14151.0   673.289542  920.496897  0.000278    0.000278     0.000278   

               75%          max  
class                            
0      2161.477500  2879.992222  
1      1330.697361  2878.874167  


## 3. Data Transformation and Imbalance Handling

if I apply SMOTE to the whole dataset before splitting, synthetic fraud examples generated from real test-set fraud cases can end up sitting in my training set, meaning my model effectively "sees" information derived from the test set during training. My test performance will then look artificially fantastic — not because my model is good, but because it's been secretly leaked a preview of the answer key. This is called data leakage, and it's exactly the kind of mistake that looks completely fine on the surface (code runs, numbers look great) but silently produces a portfolio piece that's fundamentally wrong.

#### The steps to follow:
1. Split into train/test (stratified, to preserve class ratio in both)
2. Scale/encode — fit the scaler/encoder on train only, then apply to test (same leakage principle applies here too, actually, though slightly less catastrophic)
3. Apply SMOTE/undersampling to the training set only
4. Leave the test set completely untouched in its original, real-world imbalanced form — because that's what production data will actually look like

### Split, Encoding, Scaling, and SMOTE

In [15]:
## Step 1: Preliminary train/test split
from sklearn.model_selection import train_test_split

X = fraud_df.drop(columns=[
    'class', 'signup_time', 'purchase_time', 'device_id', 'user_id',
    'time_since_signup', 'user_transaction_count'
])
y = fraud_df['class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))


class
0    0.906352
1    0.093648
Name: proportion, dtype: float64
class
0    0.906363
1    0.093637
Name: proportion, dtype: float64


In [16]:
## Step 2: Encoding (fit on train, apply to both)
X_train = pd.get_dummies(X_train, columns=['source', 'browser', 'sex', 'country'], drop_first=True)
X_test = pd.get_dummies(X_test, columns=['source', 'browser', 'sex', 'country'], drop_first=True)

# Align columns in case some category only appears in one split
X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)

print(X_train.shape, X_test.shape)


(120889, 192) (30223, 192)


In [17]:
## Step 3: Scaling numeric features
from sklearn.preprocessing import StandardScaler

numeric_cols = ['purchase_value', 'age', 'ip_address', 'time_since_signup_hours',
                 'hour_of_day', 'day_of_week', 'device_shared_count']

scaler = StandardScaler()
X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test[numeric_cols] = scaler.transform(X_test[numeric_cols])

print(X_train[numeric_cols].describe())

       purchase_value           age    ip_address  time_since_signup_hours  \
count    1.208890e+05  1.208890e+05  1.208890e+05             1.208890e+05   
mean    -2.492122e-17  1.059152e-16  5.078286e-17            -2.584989e-16   
std      1.000004e+00  1.000004e+00  1.000004e+00             1.000004e+00   
min     -1.525405e+00 -1.760092e+00 -1.724985e+00            -1.579066e+00   
25%     -8.154141e-01 -7.155302e-01 -8.527442e-01            -8.770660e-01   
50%     -1.054232e-01 -1.915541e-02  2.566300e-04            -1.158195e-03   
75%      6.591825e-01  6.772193e-01  8.735169e-01             8.662588e-01   
max      6.393725e+00  4.971530e+00  1.719512e+00             1.741269e+00   

        hour_of_day   day_of_week  device_shared_count  
count  1.208890e+05  1.208890e+05         1.208890e+05  
mean   8.416789e-17 -7.347057e-18        -4.408234e-17  
std    1.000004e+00  1.000004e+00         1.000004e+00  
min   -1.668277e+00 -1.503082e+00        -2.622236e-01  
25%   -8.001

In [18]:
## Step 4: SMOTE (Synthetic Minority Oversampling Technique)
from imblearn.over_sampling import SMOTE

print("Before SMOTE:", y_train.value_counts())

smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print("After SMOTE:", y_train_resampled.value_counts())

Before SMOTE: class
0    109568
1     11321
Name: count, dtype: int64
After SMOTE: class
0    109568
1    109568
Name: count, dtype: int64


## Finalizing the notebook with saved dataset

In [19]:
import joblib

joblib.dump((X_train, X_test, y_train, y_test), '../data/processed/fraud_train_test_split.pkl')
joblib.dump((X_train_resampled, y_train_resampled), '../data/processed/fraud_train_resampled.pkl')

['../data/processed/fraud_train_resampled.pkl']